# 003 · Silver: Data Quality

**Trabalho 3 — Lakehouse com Databricks e arquitetura medalhão** · Autores: **Murilo Mandelli** e **Augusto Benedetti**

Lê as tabelas da **bronze**, aplica regras de **qualidade de dados** e grava no schema **`silver`**.

| Código | Problema na origem | Regra aplicada na silver |
|---|---|---|
| P01 | Nome com espaços sobrando e minúsculas | `trim` + espaços duplos removidos + `initcap` |
| P02 | E-mail em MAIÚSCULAS e com espaços | `lower(trim(email))` |
| P03 | CPF com máscara (`123.456.789-01`) | Só dígitos; CPF sem 11 dígitos vira NULL |
| P04 | UF em minúsculas | `upper(trim(uf))` |
| P05 | Cidade vazia | `NAO INFORMADO` |
| P06 | Cliente duplicado (mesmo CPF) | Mantém o cadastro mais antigo (menor código) |
| P07 | Preço negativo | Valor absoluto (erro de sinal) |
| P08 | Estoque negativo | Zero |
| P09 | Status escrito de formas diferentes | `upper(trim(status))` |
| P10 | Item com quantidade zero | Registro descartado |
| P11 | Categoria sem descrição | `SEM DESCRICAO` |
| — | Linhas repetidas em qualquer tabela | `dropDuplicates()` |

Depois, como no modelo da aula, os **nomes das colunas** são padronizados (MAIÚSCULAS e siglas por
extenso: `id_` → `CODIGO_`, `dt_` → `DATA_`, `uf` → `UNIDADE_FEDERATIVA`) e são adicionadas colunas
de **auditoria**. Ao final, a tabela `silver.relatorio_data_quality` registra quantos registros
cada regra corrigiu.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = {t: spark.table(f"bronze.{t}") for t in ["categoria", "cliente", "produto", "pedido", "item_pedido"]}
relatorio = []


def registrar(tabela, codigo, regra, qtd):
    relatorio.append((tabela, codigo, regra, int(qtd)))
    print(f"{tabela:<12} {codigo:<4} {regra:<55} {qtd:>5}")


def sem_duplicadas(df, tabela):
    antes = df.count()
    df = df.dropDuplicates([c for c in df.columns if c not in ("data_hora_bronze", "nome_arquivo")])
    registrar(tabela, "-", "linhas repetidas removidas", antes - df.count())
    return df

## Regras de qualidade por tabela

In [ ]:
# ---------- categoria ----------
categoria = sem_duplicadas(bronze["categoria"], "categoria")
registrar("categoria", "P11", "descricao vazia -> SEM DESCRICAO", categoria.filter(F.col("descricao").isNull()).count())
categoria = (categoria
    .withColumn("nome", F.trim("nome"))
    .withColumn("descricao", F.coalesce(F.trim("descricao"), F.lit("SEM DESCRICAO"))))

# ---------- cliente ----------
cliente = sem_duplicadas(bronze["cliente"], "cliente")
nome_sem_espacos = F.regexp_replace(F.trim("nome"), r"\s+", " ")
# Só troca as maiúsculas/minúsculas de nomes escritos TODO em minúsculo ou TODO em maiúsculo
# (assim "Miguel da Cunha", que já está certo, não vira "Miguel Da Cunha")
nome_limpo = (F.when((nome_sem_espacos == F.lower(nome_sem_espacos)) | (nome_sem_espacos == F.upper(nome_sem_espacos)),
                     F.initcap(nome_sem_espacos))
               .otherwise(nome_sem_espacos))
registrar("cliente", "P01", "nome com espacos/minusculas -> padronizado",
          cliente.filter(F.col("nome") != nome_limpo).count())
registrar("cliente", "P02", "email maiusculo/espacos -> minusculo",
          cliente.filter(F.col("email") != F.lower(F.trim("email"))).count())
registrar("cliente", "P03", "cpf com mascara -> so digitos",
          cliente.filter(F.col("cpf").cast("string").rlike(r"\D")).count())
registrar("cliente", "P04", "uf minuscula -> maiuscula",
          cliente.filter(F.col("uf") != F.upper(F.trim("uf"))).count())
registrar("cliente", "P05", "cidade vazia -> NAO INFORMADO", cliente.filter(F.col("cidade").isNull()).count())

cpf_digitos = F.regexp_replace(F.col("cpf").cast("string"), r"\D", "")
cliente = (cliente
    .withColumn("nome", nome_limpo)
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("cpf", F.when(F.length(cpf_digitos).between(9, 11), F.lpad(cpf_digitos, 11, "0")))
    .withColumn("uf", F.upper(F.trim("uf")))
    .withColumn("cidade", F.coalesce(F.trim("cidade"), F.lit("NAO INFORMADO"))))

# P06: mesmo CPF cadastrado mais de uma vez -> fica o cadastro mais antigo (menor id)
antes = cliente.count()
ordem = Window.partitionBy("cpf").orderBy("id_cliente")
cliente = cliente.withColumn("_n", F.row_number().over(ordem)).filter("_n = 1").drop("_n")
registrar("cliente", "P06", "cadastro duplicado (mesmo CPF) removido", antes - cliente.count())

In [ ]:
# ---------- produto ----------
produto = sem_duplicadas(bronze["produto"], "produto")
registrar("produto", "P07", "preco negativo -> valor absoluto", produto.filter("preco < 0").count())
registrar("produto", "P08", "estoque negativo -> zero", produto.filter("estoque < 0").count())
produto = (produto
    .withColumn("nome", F.trim("nome"))
    .withColumn("preco", F.abs("preco"))
    .withColumn("estoque", F.greatest(F.col("estoque"), F.lit(0))))

# ---------- pedido ----------
pedido = sem_duplicadas(bronze["pedido"], "pedido")
registrar("pedido", "P09", "status fora do padrao -> MAIUSCULO sem espacos",
          pedido.filter(F.col("status") != F.upper(F.trim("status"))).count())
pedido = pedido.withColumn("status", F.upper(F.trim("status")))

# ---------- item_pedido ----------
item_pedido = sem_duplicadas(bronze["item_pedido"], "item_pedido")
registrar("item_pedido", "P10", "quantidade zero ou negativa -> descartado", item_pedido.filter("quantidade <= 0").count())
item_pedido = item_pedido.filter("quantidade > 0")

## Padronizar nomes de colunas, auditoria e gravar na silver

Mesmo padrão do notebook da aula: nomes em MAIÚSCULAS, siglas por extenso, remoção das colunas da
bronze e inclusão de `NOME_TABELA_BRONZE` e `DATA_HORA_SILVER`.

In [ ]:
def padronizar_nome(coluna):
    n = coluna.upper()
    if n == "UF":
        return "UNIDADE_FEDERATIVA"
    for sigla, extenso in {"ID_": "CODIGO_", "DT_": "DATA_", "VL_": "VALOR_", "CD_": "CODIGO_"}.items():
        if n.startswith(sigla):
            n = extenso + n[len(sigla):]
    return n


def gravar_silver(df, tabela):
    df = df.drop("data_hora_bronze", "nome_arquivo")
    df = df.toDF(*[padronizar_nome(c) for c in df.columns])
    df = (df.withColumn("NOME_TABELA_BRONZE", F.lit(f"bronze.{tabela}"))
            .withColumn("DATA_HORA_SILVER", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"silver.{tabela}"))
    print(f"silver.{tabela:<12} {df.count():>6} registros | {df.columns}")


gravar_silver(categoria, "categoria")
gravar_silver(cliente, "cliente")
gravar_silver(produto, "produto")
gravar_silver(pedido, "pedido")
gravar_silver(item_pedido, "item_pedido")

## Relatório de Data Quality

In [ ]:
(spark.createDataFrame(relatorio, "TABELA string, CODIGO string, REGRA string, REGISTROS_AFETADOS int")
      .withColumn("DATA_HORA_SILVER", F.current_timestamp())
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("silver.relatorio_data_quality"))

In [ ]:
%sql
SELECT TABELA, CODIGO, REGRA, REGISTROS_AFETADOS
FROM silver.relatorio_data_quality
WHERE REGISTROS_AFETADOS > 0
ORDER BY CODIGO

Os mesmos clientes da bronze, agora corrigidos:

In [ ]:
%sql
SELECT CODIGO_CLIENTE, NOME, EMAIL, CPF, CIDADE, UNIDADE_FEDERATIVA
FROM silver.cliente
ORDER BY CODIGO_CLIENTE
LIMIT 10

In [ ]:
%sql
SELECT STATUS, COUNT(*) AS QTD FROM silver.pedido GROUP BY STATUS ORDER BY STATUS

In [ ]:
%sql
SHOW TABLES IN silver